## Explicación del Algoritmo Minimax y Poda Alfa-Beta

### Algoritmo Minimax

El algoritmo Minimax es una técnica de toma de decisiones utilizada en inteligencia artificial para juegos de dos jugadores con información completa y turnos alternos, como el Tic-Tac-Toe. Su objetivo es encontrar el mejor movimiento posible para un jugador, asumiendo que el oponente también juega de manera óptima.

El algoritmo funciona construyendo un árbol de juego que representa todas las posibles secuencias de movimientos desde el estado actual hasta el final del juego. Luego, evalúa los estados finales del juego (hojas del árbol) asignándoles un valor numérico:

*   **Valor alto:** Si el jugador actual (la IA en este caso) gana.
*   **Valor bajo:** Si el oponente gana.
*   **Valor cero:** Si es un empate.

Estos valores se propagan hacia arriba en el árbol. En los nodos que representan el turno del jugador actual (maximizador), se elige el movimiento que lleva al estado con el valor más alto. En los nodos que representan el turno del oponente (minimizador), se asume que el oponente elegirá el movimiento que lleva al estado con el valor más bajo para el jugador actual.

Finalmente, el mejor movimiento para el jugador actual es aquel que maximiza su ganancia mínima posible, asumiendo que el oponente juega perfectamente para minimizar esa ganancia.

### Poda Alfa-Beta

La poda Alfa-Beta es una optimización del algoritmo Minimax que reduce el número de nodos que deben explorarse en el árbol de juego. Mantiene la misma decisión que Minimax puro, pero es mucho más eficiente.

Utiliza dos valores, Alfa (α) y Beta (β), para "podar" ramas del árbol que no es necesario explorar:

*   **Alfa (α):** Representa el mejor valor que el jugador maximizador (la IA) puede garantizar hasta el momento en la rama actual.
*   **Beta (β):** Representa el mejor valor que el jugador minimizador (el oponente) puede garantizar hasta el momento en la rama actual.

La poda ocurre cuando:

*   En un turno del jugador minimizador, si β (el mejor valor que el oponente puede asegurar) es menor o igual que α (el mejor valor que la IA ya ha asegurado en otra rama), la rama actual se "poda" porque el oponente ya tiene una opción mejor y no elegirá esta rama.
*   En un turno del jugador maximizador, si α (el mejor valor que la IA puede asegurar) es mayor o igual que β (el mejor valor que el oponente puede asegurar en otra rama), la rama actual se "poda" porque la IA ya tiene una opción mejor en otro lugar.

La poda Alfa-Beta elimina la necesidad de explorar ramas del árbol que se sabe que no afectarán el resultado final, lo que acelera significativamente el proceso de toma de decisiones de la IA en juegos complejos.

In [1]:
board = [' '] * 9 # Represents the Tic-Tac-Toe board as a list of 9 strings

def display_board(board):
    """Displays the Tic-Tac-Toe board."""
    print('-------------')
    print('|', board[0], '|', board[1], '|', board[2], '|')
    print('-------------')
    print('|', board[3], '|', board[4], '|', board[5], '|')
    print('-------------')
    print('|', board[6], '|', board[7], '|', board[8], '|')
    print('-------------')

def check_win(board, mark):
    """Checks if a player has won."""
    # Define all possible winning combinations
    win_combinations = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],  # Rows
        [0, 3, 6], [1, 4, 7], [2, 5, 8],  # Columns
        [0, 4, 8], [2, 4, 6]   # Diagonals
    ]
    # Check if any winning combination is present on the board with the given mark
    for combo in win_combinations:
        if board[combo[0]] == board[combo[1]] == board[combo[2]] == mark:
            return True
    return False

def check_draw(board):
    """Checks if the board is full (draw)."""
    # Check if there are no empty spaces and no player has won
    return ' ' not in board and not check_win(board, 'X') and not check_win(board, 'O')

def make_move(board, mark, position):
    """Places the player's mark on the board if the position is valid."""
    # Check if the position is within the valid range (0-8) and the space is empty
    if 0 <= position < 9 and board[position] == ' ':
        board[position] = mark # Place the mark on the board
        return True # Indicate that the move was successful
    return False # Indicate that the move was invalid

In [2]:
def evaluate(board):
    """Evaluates the current state of the board for the Minimax algorithm."""
    if check_win(board, 'X'):
        return 10  # AI (X) wins
    elif check_win(board, 'O'):
        return -10 # Human (O) wins
    elif check_draw(board):
        return 0   # Draw
    else:
        return 0   # Game is ongoing

def minimax(board, depth, is_maximizing, alpha, beta):
    """
    Implements the Minimax algorithm with Alpha-Beta pruning.

    Args:
        board: The current state of the Tic-Tac-Toe board.
        depth: The current depth of the search tree.
        is_maximizing: True if it's the maximizing player's turn (AI), False otherwise.
        alpha: The best value found so far for the maximizing player.
        beta: The best value found so far for the minimizing player.

    Returns:
        The optimal value for the current board state.
    """
    score = evaluate(board)

    # Base cases: game is over or a win/loss state is reached
    if score == 10:
        return score - depth # Prioritize winning faster
    if score == -10:
        return score + depth # Prioritize losing slower
    if check_draw(board):
        return 0

    if is_maximizing: # AI's turn (maximizing player)
        max_eval = float('-inf')
        for i in range(9):
            if board[i] == ' ': # Check for empty spots
                board[i] = 'X' # Make a hypothetical move
                eval = minimax(board, depth + 1, False, alpha, beta) # Recursively call for the next player
                board[i] = ' ' # Undo the hypothetical move
                max_eval = max(max_eval, eval) # Update the maximum evaluation
                alpha = max(alpha, eval) # Update alpha
                if beta <= alpha: # Alpha-Beta pruning condition
                    break
        return max_eval
    else: # Human's turn (minimizing player)
        min_eval = float('inf')
        for i in range(9):
            if board[i] == ' ': # Check for empty spots
                board[i] = 'O' # Make a hypothetical move
                eval = minimax(board, depth + 1, True, alpha, beta) # Recursively call for the next player
                board[i] = ' ' # Undo the hypothetical move
                min_eval = min(min_eval, eval) # Update the minimum evaluation
                beta = min(beta, eval) # Update beta
                if beta <= alpha: # Alpha-Beta pruning condition
                    break
        return min_eval

def find_best_move(board):
    """Finds the best move for the AI using the Minimax algorithm."""
    best_eval = float('inf') # Initialize with a very large value
    best_move = -1 # Initialize with an invalid move
    alpha = float('-inf') # Initialize alpha for Alpha-Beta pruning
    beta = float('inf')  # Initialize beta for Alpha-Beta pruning
    for i in range(9):
        if board[i] == ' ': # Check for empty spots
            board[i] = 'O' # Make a hypothetical move for the AI
            eval = minimax(board, 0, True, alpha, beta) # Evaluate the move using Minimax (starting with the maximizing player's turn)
            board[i] = ' ' # Undo the hypothetical move
            if eval < best_eval: # If the current move's evaluation is better than the best found so far
                best_eval = eval # Update the best evaluation
                best_move = i # Update the best move
            beta = min(beta, eval) # Update beta after evaluating a move
    return best_move # Return the index of the best move for the AI

In [3]:
while True:
    display_board(board) # Display the current state of the board

    # Human player's turn
    while True:
        try:
            move = int(input("Enter your move (0-8): ")) # Get the human player's move input
            if make_move(board, 'X', move): # Attempt to make the move
                break # If the move is valid, exit the input loop
            else:
                print("Invalid move. Try again.") # If the move is invalid, prompt again
        except ValueError:
            print("Invalid input. Please enter a number.") # Handle non-integer input

    # Check for human win
    if check_win(board, 'X'):
        display_board(board) # Display the final board
        print("You win!") # Announce human win
        break # Exit the main game loop

    # Check for draw
    if check_draw(board):
        display_board(board) # Display the final board
        print("It's a draw!") # Announce draw
        break # Exit the main game loop

    # AI player's turn
    print("AI is making a move...") # Indicate AI's turn
    ai_move = find_best_move(board) # Get the AI's best move using Minimax
    make_move(board, 'O', ai_move) # Make the AI's move

    # Check for AI win
    if check_win(board, 'O'):
        display_board(board) # Display the final board
        print("AI wins!") # Announce AI win
        break # Exit the main game loop

    # Check for draw
    if check_draw(board):
        display_board(board) # Display the final board
        print("It's a draw!") # Announce draw
        break # Exit the main game loop

-------------
|   |   |   |
-------------
|   |   |   |
-------------
|   |   |   |
-------------
Enter your move (0-8): 5
AI is making a move...
-------------
|   |   | O |
-------------
|   |   | X |
-------------
|   |   |   |
-------------
Enter your move (0-8): 4
AI is making a move...
-------------
|   |   | O |
-------------
| O | X | X |
-------------
|   |   |   |
-------------
Enter your move (0-8): 3
Invalid move. Try again.
Enter your move (0-8): 2
Invalid move. Try again.
Enter your move (0-8): 0
AI is making a move...
-------------
| X |   | O |
-------------
| O | X | X |
-------------
|   |   | O |
-------------
Enter your move (0-8): 2
Invalid move. Try again.
Enter your move (0-8): 0
Invalid move. Try again.
Enter your move (0-8): 1
AI is making a move...
-------------
| X | X | O |
-------------
| O | X | X |
-------------
|   | O | O |
-------------
Enter your move (0-8): 6
-------------
| X | X | O |
-------------
| O | X | X |
-------------
| X | O | O |
---------